# SQL Business Analysis — UCI Online Retail 

## Overview

This notebook uses SQL (via DuckDB) to answer four business questions about customer behaviour in the UCI Online Retail dataset.

It builds directly on the RFM + KMeans segmentation in `01_customer_analytics.ipynb`, importing cluster labels to cross-analyse retention and purchase behaviour at the segment level.

**Four business questions, in order of analytical dependency:**

| # | Question | Why it matters |
|---|----------|---------------|
| 1 | What share of customers ever return? | Establishes the baseline retention problem |
| 2 | When do customers return — and when do they stop? | Identifies the intervention window |
| 3 | Where in the purchase funnel does the business lose customers? | Quantifies the steepest drop-off point |
| 4 | Do different customer segments behave differently — and what does that imply for tactics? | Translates the population-level findings into segment-specific actions |

Each case builds on the previous one. By Case 4, the analysis has enough context to recommend specific, data-grounded tactics for each segment — not generic best practices.


## Dataset

The dataset contains transactional records from a UK-based online retailer, covering December 2010 – December 2011.

| Field | Description |
|---|---|
| `CustomerID` | Unique customer identifier |
| `InvoiceNo` | Transaction ID (prefix `C` = cancellation) |
| `StockCode` | Product code |
| `Description` | Product name |
| `Quantity` | Units per line item |
| `UnitPrice` | Price per unit (GBP) |
| `InvoiceDate` | Timestamp of transaction |
| `Country` | Customer country |

Each row represents a single product line within a transaction.

## Setup

Connect to DuckDB and load the dataset. Rows with missing `CustomerID`, cancelled orders (InvoiceNo starting with `C`), or non-positive Quantity / UnitPrice are removed before analysis.

In [19]:
import duckdb
import pandas as pd

con = duckdb.connect()

In [20]:
df = pd.read_csv("online_retail.csv")


In [21]:
df.shape

(541909, 8)

In [22]:
df_clean = df.copy()

# remove missing customer
df_clean = df_clean.dropna(subset=['CustomerID'])

# remove cancelled orders
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]

# remove invalid values
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]

df_clean.shape


(397884, 8)

In [23]:
con.register("retail", df_clean)

---

## Case 1: Overall Repeat Purchase Rate

### Business question
What share of customers ever make a second purchase?

A high repeat rate suggests the business builds loyalty. A low rate suggests it runs on acquisition — expensive and fragile. This is the starting point before asking *when* or *which customers* return.


In [24]:
query = """
WITH customer_orders AS (
    SELECT 
        CustomerID,
        COUNT(DISTINCT DATE(InvoiceDate)) AS order_count
    FROM retail
    GROUP BY CustomerID
)

SELECT 
    COUNT(*) FILTER (WHERE order_count > 1) * 1.0 
        / COUNT(*) AS repeat_purchase_rate
FROM customer_orders
"""

con.execute(query).fetchdf()

,repeat_purchase_rate
0,0.643154


In [25]:
query = """
WITH first_purchase AS (
    SELECT 
        CustomerID,
        MIN(DATE(InvoiceDate)) AS first_date
    FROM retail
    GROUP BY CustomerID
),

customer_orders AS (
    SELECT 
        r.CustomerID,
        COUNT(DISTINCT DATE(r.InvoiceDate)) AS active_days,
        SUM(CASE WHEN DATE(r.InvoiceDate) > f.first_date THEN 1 ELSE 0 END) AS repeat_activity
    FROM retail r
    JOIN first_purchase f
        ON r.CustomerID = f.CustomerID
    GROUP BY r.CustomerID
)

SELECT 
    COUNT(*) FILTER (WHERE repeat_activity > 0) * 1.0 
        / COUNT(*) AS repeat_purchase_rate
FROM customer_orders


"""

con.execute(query).fetchdf()

,repeat_purchase_rate
0,0.643154


**64.3% of customers make at least one repeat purchase** across the full observation window (Dec 2010 – Dec 2011).

This looks healthy — but it is a lifetime metric. A customer who returned once in 12 months counts the same as one who returned every month. The next question is *when* they return, because timing determines whether intervention is even possible.


---

## Case 2: Return Timing — When Do Customers Come Back?

### Business question
Of the customers who do return, when do they return? Is there a window where the business can realistically intervene?

If most customers return within 30 days naturally, a coupon adds little value. If the return window extends to 60–90 days, there is a meaningful intervention opportunity — but only if the nudge arrives before the customer has fully lapsed.


### Time-windowed repeat rate (30 / 60 / 90 days post first purchase)

In [26]:
query = """
WITH first_purchase AS (
    SELECT 
        CustomerID,
        MIN(DATE(InvoiceDate)) AS first_date
    FROM retail
    GROUP BY CustomerID
),

purchases AS (
    SELECT 
        r.CustomerID,
        DATE(r.InvoiceDate) AS purchase_date,
        f.first_date
    FROM retail r
    JOIN first_purchase f
        ON r.CustomerID = f.CustomerID
),

flags AS (
    SELECT 
        CustomerID,

        MAX(CASE 
            WHEN purchase_date > first_date 
             AND purchase_date <= first_date + INTERVAL 30 DAY 
            THEN 1 ELSE 0 END) AS repeat_30d,

        MAX(CASE 
            WHEN purchase_date > first_date 
             AND purchase_date <= first_date + INTERVAL 60 DAY 
            THEN 1 ELSE 0 END) AS repeat_60d,

        MAX(CASE 
            WHEN purchase_date > first_date 
             AND purchase_date <= first_date + INTERVAL 90 DAY 
            THEN 1 ELSE 0 END) AS repeat_90d

    FROM purchases
    GROUP BY CustomerID
)

SELECT 
    COUNT(*) FILTER (WHERE repeat_30d = 1) * 1.0 / COUNT(*) AS repeat_30d_rate,
    COUNT(*) FILTER (WHERE repeat_60d = 1) * 1.0 / COUNT(*) AS repeat_60d_rate,
    COUNT(*) FILTER (WHERE repeat_90d = 1) * 1.0 / COUNT(*) AS repeat_90d_rate
FROM flags
"""
con.execute(query).fetchdf()

,repeat_30d_rate,repeat_60d_rate,repeat_90d_rate
0,0.187183,0.335639,0.422084


| Window | Cumulative return rate | Incremental gain |
|--------|----------------------|-----------------|
| 0 – 30 days | 18.7% | baseline |
| 0 – 60 days | 33.6% | **+14.9 pp** ← largest single jump |
| 0 – 90 days | 42.2% | +8.6 pp |

**The 31–60 day window captures nearly twice the incremental return rate of the 61–90 day window.** This is not random — it aligns with the inter-purchase gap distribution below.


In [27]:
# csv for tableau
retention_windows = pd.DataFrame({
    'Window': ['0-30d', '0-60d', '0-90d'],
    'Rate': [0.187, 0.336, 0.422],
    'Incremental': [0.187, 0.149, 0.086]
})
retention_windows.to_csv('retention_windows.csv', index=False)

### Inter-purchase gap distribution (all repeat buyers)

In [28]:
query = """
WITH order_dates AS (
    SELECT
        CAST(CustomerID AS INTEGER) AS CustomerID,
        DATE(InvoiceDate) AS purchase_date,
        LAG(DATE(InvoiceDate)) OVER (
            PARTITION BY CAST(CustomerID AS INTEGER) 
            ORDER BY DATE(InvoiceDate)
        ) AS prev_date
    FROM retail
    WHERE CustomerID IS NOT NULL
      AND Quantity > 0
      AND UnitPrice > 0
    GROUP BY CustomerID, DATE(InvoiceDate)
)
SELECT
    ROUND(AVG(DATEDIFF('day', prev_date, purchase_date)), 1) AS avg_days_between_purchases,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (
        ORDER BY DATEDIFF('day', prev_date, purchase_date)
    ), 1) AS median_days,
    MIN(DATEDIFF('day', prev_date, purchase_date)) AS min_days,
    MAX(DATEDIFF('day', prev_date, purchase_date)) AS max_days
FROM order_dates
WHERE prev_date IS NOT NULL
"""

con.execute(query).fetchdf()

,avg_days_between_purchases,median_days,min_days,max_days
0,45.7,28.0,1,366


The gap distribution is right-skewed: **median = 28 days, mean = 45.7 days**.

The gap between median and mean tells a specific story: most customers who return do so within 28 days (they didn't need a nudge), but a long tail of slow returners pulls the mean up to 45.7 days. This tail is the addressable segment — customers who *might* return if prompted, but won't do so on their own within 30 days.

**Implication for coupon design (population level):**
- A 30-day expiry fires before the slow returners have had time to decide → wasted
- A 90-day expiry removes urgency → customers procrastinate and conversion drops
- A **45-day expiry** sits inside the peak return window (day 31–60) and aligns with the mean gap — it creates urgency at exactly the point where the marginal customer is most persuadable

> This is the *population-level* rationale. Whether the optimal window differs by segment is tested in Case 4 and validated statistically in `03_ab_test_simulation.ipynb`.


### Purchase frequency distribution

In [29]:
query = """
WITH customer_orders AS (
    SELECT 
        CustomerID,
        COUNT(DISTINCT DATE(InvoiceDate)) AS active_days
    FROM retail
    GROUP BY CustomerID
)

SELECT 
    CASE 
        WHEN active_days = 1 THEN '1 purchase'
        WHEN active_days = 2 THEN '2 purchases'
        WHEN active_days <= 5 THEN '3-5 purchases'
        ELSE '6+ purchases'
    END AS purchase_bucket,
    COUNT(*) AS customers
FROM customer_orders
GROUP BY purchase_bucket
ORDER BY purchase_bucket
"""
con.execute(query).fetchdf()

,purchase_bucket,customers
0,1 purchase,1548
1,2 purchases,874
2,3-5 purchases,1117
3,6+ purchases,799


1,548 customers (35.7%) bought exactly once and never returned. This is the primary target population for any win-back campaign. The 874 two-time buyers and 1,117 three-to-five-time buyers represent customers who are already in the habit of returning — intervention here has lower marginal value.


---

## Case 3: Purchase Funnel — Where Does the Business Lose Customers?

### Business question
At each successive purchase, what share of customers continue? Is the drop-off concentrated at one step, or distributed evenly across the funnel?

This is a structural question. If churn is concentrated at one step, that step is the highest-leverage intervention point. If it is distributed evenly, the problem is systemic and harder to address with a single tactic.


In [30]:
query = """
WITH order_sequence AS (
    SELECT
        CAST(CustomerID AS INTEGER) AS CustomerID,
        InvoiceNo,
        DATE(InvoiceDate) AS purchase_date,
        ROW_NUMBER() OVER (
            PARTITION BY CAST(CustomerID AS INTEGER) 
            ORDER BY DATE(InvoiceDate)
        ) AS purchase_rank
    FROM retail
    WHERE CustomerID IS NOT NULL 
      AND Quantity > 0
    GROUP BY CustomerID, InvoiceNo, DATE(InvoiceDate)
)
SELECT
    COUNT(DISTINCT CASE WHEN purchase_rank >= 1 THEN CustomerID END) AS made_1st,
    COUNT(DISTINCT CASE WHEN purchase_rank >= 2 THEN CustomerID END) AS made_2nd,
    COUNT(DISTINCT CASE WHEN purchase_rank >= 3 THEN CustomerID END) AS made_3rd,
    COUNT(DISTINCT CASE WHEN purchase_rank >= 4 THEN CustomerID END) AS made_4th,
    COUNT(DISTINCT CASE WHEN purchase_rank >= 5 THEN CustomerID END) AS made_5th,

    ROUND(COUNT(DISTINCT CASE WHEN purchase_rank >= 2 THEN CustomerID END) * 100.0 /
          COUNT(DISTINCT CASE WHEN purchase_rank >= 1 THEN CustomerID END), 1) AS "1→2_%",
    ROUND(COUNT(DISTINCT CASE WHEN purchase_rank >= 3 THEN CustomerID END) * 100.0 /
          COUNT(DISTINCT CASE WHEN purchase_rank >= 2 THEN CustomerID END), 1) AS "2→3_%",
    ROUND(COUNT(DISTINCT CASE WHEN purchase_rank >= 4 THEN CustomerID END) * 100.0 /
          COUNT(DISTINCT CASE WHEN purchase_rank >= 3 THEN CustomerID END), 1) AS "3→4_%",
    ROUND(COUNT(DISTINCT CASE WHEN purchase_rank >= 5 THEN CustomerID END) * 100.0 /
          COUNT(DISTINCT CASE WHEN purchase_rank >= 4 THEN CustomerID END), 1) AS "4→5_%"
FROM order_sequence
"""

con.execute(query).fetchdf()

,made_1st,made_2nd,made_3rd,made_4th,made_5th,1→2_%,2→3_%,3→4_%,4→5_%
0,4338,2845,2010,1502,1114,65.6,70.7,74.7,74.2


| Step | Customers remaining | Step conversion | Drop-off |
|------|--------------------|-----------------| --------|
| 1st purchase | 4,338 | — | — |
| 2nd purchase | 2,845 | 65.6% | **34.4% lost** ← steepest drop |
| 3rd purchase | 2,010 | 70.7% | 29.3% lost |
| 4th purchase | 1,502 | 74.7% | 25.3% lost |
| 5th purchase | 1,114 | 74.2% | 25.8% lost |

**The 1→2 transition loses 34.4% of customers — nearly 10 percentage points more than any subsequent step.** From the 3rd purchase onward, conversion stabilises in the 70–75% range.

This is a classic new-customer activation problem. The business is good at retaining customers who have already made two purchases — the failure point is converting the first purchase into a habit. Every tactic in Cases 1 and 2 ultimately points here: the 45-day coupon is only valuable insofar as it closes this specific gap.


In [31]:
#export csv for tableau
import pandas as pd

data = {
    'Purchase': ['1st', '2nd', '3rd', '4th', '5th'],
    'Customers': [4338, 2845, 2010, 1502, 1114],
    'Offset': [-(4338/2), -(2845/2), -(2010/2), -(1502/2), -(1114/2)]
}
df = pd.DataFrame(data)
df.to_csv('funnel_data.csv', index=False)
print(df)

  Purchase  Customers  Offset
0      1st       4338 -2169.0
1      2nd       2845 -1422.5
2      3rd       2010 -1005.0
3      4th       1502  -751.0
4      5th       1114  -557.0


---

## Case 4: Behavioural Analysis by Segment — What Should We Actually Do?

### Business question
Do the four RFM segments differ meaningfully in their purchase behaviour — and if so, does the population-level 45-day coupon recommendation still hold, or does each segment need a different approach?

Cases 1–3 treated all 4,338 customers as a single population. That is useful for identifying the problem but not for designing the solution. A Whale customer (avg 82 orders, recency 7 days) and an At-risk customer (avg 1.5 orders, recency 248 days) are not the same problem — and should not receive the same intervention.


### Load cluster labels

In [32]:
# Load RFM cluster labels from the segmentation notebook
# Register in DuckDB
con.execute("CREATE TABLE rfm_clusters AS SELECT * FROM read_csv_auto('rfm_clusters.csv')")

# Verify
con.execute("SELECT * FROM rfm_clusters LIMIT 5").fetchdf()

,CustomerID,Cluster
0,12346.0,3
1,12347.0,0
2,12348.0,0
3,12349.0,0
4,12350.0,1


### Full RFM + behavioural profile by segment

In [33]:
query = """
WITH retail_stats AS (
    SELECT
        CAST(CustomerID AS INTEGER) AS CustomerID,
        COUNT(DISTINCT InvoiceNo)                                          AS num_orders,
        ROUND(SUM(Quantity * UnitPrice), 2)                                AS total_revenue,
        ROUND(SUM(Quantity * UnitPrice) / COUNT(DISTINCT InvoiceNo), 2)    AS avg_order_value,
        COUNT(DISTINCT StockCode)                                          AS unique_products,
        COUNT(DISTINCT DATE(InvoiceDate))                                  AS active_days,
        MIN(DATE(InvoiceDate))                                             AS first_purchase,
        MAX(DATE(InvoiceDate))                                             AS last_purchase,
        DATEDIFF('day', MIN(DATE(InvoiceDate)), MAX(DATE(InvoiceDate)))    AS customer_lifespan_days
    FROM retail
    WHERE Quantity > 0 AND UnitPrice > 0 AND CustomerID IS NOT NULL
    GROUP BY CustomerID
),
clusters AS (
    SELECT CAST(CAST(CustomerID AS FLOAT) AS INTEGER) AS CustomerID, Cluster
    FROM rfm_clusters
),
combined AS (
    SELECT
        CASE c.Cluster
            WHEN 2 THEN 'C2: Whale'
            WHEN 3 THEN 'C3: High-value loyal'
            WHEN 0 THEN 'C0: Regular'
            WHEN 1 THEN 'C1: At-risk / lapsed'
        END AS segment,
        c.Cluster,
        r.*
    FROM retail_stats r
    JOIN clusters c ON r.CustomerID = c.CustomerID
)
SELECT
    segment,
    COUNT(*)                                        AS customers,
    ROUND(AVG(total_revenue), 0)                    AS avg_revenue,
    ROUND(AVG(avg_order_value), 0)                  AS avg_aov,
    ROUND(AVG(num_orders), 1)                       AS avg_orders,
    ROUND(AVG(unique_products), 0)                  AS avg_unique_products,
    ROUND(AVG(active_days), 1)                      AS avg_active_days,
    ROUND(AVG(customer_lifespan_days), 0)           AS avg_lifespan_days,
    ROUND(AVG(CASE WHEN num_orders > 1
        THEN customer_lifespan_days * 1.0 / (num_orders - 1)
        ELSE NULL END), 0)                          AS avg_gap_between_orders
FROM combined
GROUP BY segment, Cluster
ORDER BY avg_revenue DESC
"""

con.execute(query).fetchdf()


,segment,customers,avg_revenue,avg_aov,avg_orders,avg_unique_products,avg_active_days,avg_lifespan_days,avg_gap_between_orders
0,C2: Whale,13,127338.0,8571.0,82.5,670.0,56.8,348.0,21.0
1,C3: High-value loyal,204,12709.0,1080.0,22.3,185.0,19.0,333.0,18.0
2,C0: Regular,3054,1359.0,377.0,3.7,63.0,3.5,152.0,80.0
3,C1: At-risk / lapsed,1067,481.0,315.0,1.6,26.0,1.5,28.0,61.0


Reading this table across all columns reveals the distinct behavioural signature of each segment:

| Segment | Customers | Avg Revenue | Avg AOV | Avg Orders | Avg Gap (days) | Avg Recency (days) | Interpretation |
|---------|-----------|-------------|---------|------------|----------------|--------------------|----------------|
| C2: Whale | 13 | £127,338 | £8,571 | 82.5 | 21 | 7 | Recency (7d) is well inside the normal purchase cycle (21d) — these customers are almost certainly still active. The risk here is not activation but relationship management: losing a single Whale is immediately visible in revenue. No coupon; dedicated account contact. |
| C3: High-value loyal | 204 | £12,709 | £1,080 | 22.3 | 18 | 15.5 | Recency (15.5d) closely matches gap (18d) — on schedule. High-frequency, high-AOV buyers who have already established a purchase habit. They do not need to be reactivated; they need a reason not to leave. VIP programme, early access, or loyalty rewards. |
| C0: Regular | 3,054 | £1,359 | £377 | 3.7 | 80 | 43.7 | Recency (43.7d) is roughly halfway through their typical purchase cycle (80d). They are not overdue — they are mid-cycle. A 30-day post-first-purchase coupon would expire before they are even thinking about returning. A **45–60 day coupon window** better aligns with the point in their cycle when the purchase decision becomes active. |
| C1: At-risk / lapsed | 1,067 | £481 | £315 | 1.6 | 61 | 248 | Recency (248d) is 4× the historical gap (61d) — structurally lapsed. Note: avg lifespan is only 28 days and avg orders only 1.6, meaning most C1 customers have a single purchase with no gap history. The 61-day gap is derived from the minority who bought more than once, so it is directional rather than precise. A **45-day win-back coupon** aligns with their historical rhythm and creates urgency; prioritise the 60–90 day lapsed sub-segment first where reactivation cost is lowest. |

**The key differentiator between C0 and C1 is not AOV or order count — it is the ratio of current recency to historical gap.** C0 customers are mid-cycle and approaching their natural return point. C1 customers are 4× beyond their normal cycle and have structurally disengaged. The same coupon window cannot serve both.

> Note: the gap figures above use avg_gap_between_orders from the lifespan-based approximation. The LAG()-based per-cluster gap distribution in the query below provides a more precise view.

### Segment-specific purchase gap analysis

The avg_gap_between_orders above is approximate (lifespan / orders). A more precise view uses LAG() to calculate actual inter-purchase gaps per cluster.


In [34]:
query = """
WITH order_dates AS (
    SELECT
        CAST(r.CustomerID AS INTEGER) AS CustomerID,
        DATE(r.InvoiceDate) AS purchase_date,
        LAG(DATE(r.InvoiceDate)) OVER (
            PARTITION BY CAST(r.CustomerID AS INTEGER)
            ORDER BY DATE(r.InvoiceDate)
        ) AS prev_date,
        CASE c.Cluster
            WHEN 2 THEN 'C2: Whale'
            WHEN 3 THEN 'C3: High-value loyal'
            WHEN 0 THEN 'C0: Regular'
            WHEN 1 THEN 'C1: At-risk / lapsed'
        END AS segment
    FROM retail r
    JOIN (
        SELECT CAST(CAST(CustomerID AS FLOAT) AS INTEGER) AS CustomerID, Cluster
        FROM rfm_clusters
    ) c ON CAST(r.CustomerID AS INTEGER) = c.CustomerID
    WHERE r.CustomerID IS NOT NULL AND r.Quantity > 0 AND r.UnitPrice > 0
    GROUP BY r.CustomerID, DATE(r.InvoiceDate), c.Cluster
),
gaps AS (
    SELECT
        segment,
        DATEDIFF('day', prev_date, purchase_date) AS gap_days
    FROM order_dates
    WHERE prev_date IS NOT NULL
)
SELECT
    segment,
    COUNT(*)                                                                AS gap_observations,
    ROUND(AVG(gap_days), 1)                                                 AS mean_gap,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY gap_days), 1)        AS median_gap,
    ROUND(PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY gap_days), 1)       AS p25_gap,
    ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY gap_days), 1)       AS p75_gap
FROM gaps
GROUP BY segment
ORDER BY median_gap
"""

con.execute(query).fetchdf()


,segment,gap_observations,mean_gap,median_gap,p25_gap,p75_gap
0,C2: Whale,726,6.2,3.0,2.0,7.0
1,C3: High-value loyal,3673,18.5,14.0,7.0,25.0
2,C0: Regular,7522,61.8,43.0,21.0,81.0
3,C1: At-risk / lapsed,504,59.1,48.0,21.0,85.0


This per-cluster gap distribution is the direct data input for coupon window design:

| Segment | Mean gap | Median gap | P25 | P75 | Implication |
|---------|----------|------------|-----|-----|-------------|
| C2: Whale | 6.2d | 3d | 2d | 7d | Near-daily purchasing — B2B replenishment behaviour. Coupons irrelevant; these customers are structurally active. |
| C3: High-value loyal | 18.5d | 14d | 7d | 25d | Bi-weekly buying habit. Recency (15.5d) sits inside the median gap — still on cycle. No coupon needed. |
| C0: Regular | 61.8d | 43d | 21d | 81d | Wide distribution (P25=21d, P75=81d) — a heterogeneous group. Half return within 43 days naturally; the other half stretches well beyond. A **45–60 day coupon window** captures the slow-return half without being premature for the fast returners. |
| C1: At-risk / lapsed | 59.1d | 48d | 21d | 85d | Nearly identical gap distribution to C0 — C1 customers did not have a fundamentally different purchase rhythm when active. Their problem is not cadence; it is disengagement. Recency of 248 days means they are 5× beyond their normal cycle. A **45-day win-back coupon** aligns with their historical median and forces a decision before the window closes. |

**The C0 vs C1 comparison is instructive:** their gap distributions are almost identical (median 43d vs 48d), yet their recency is radically different (44d vs 248d). The segments are not separated by how often they *used to* buy — they are separated by whether they are *still* buying. The intervention logic follows from this: C0 needs a nudge mid-cycle; C1 needs a reason to restart entirely.

> **This analysis establishes the rationale for segment-specific coupon windows.** The statistical validation — whether the assumed coupon lift is real and not due to chance — is tested in `03_ab_test_simulation.ipynb` using simulated A/B experiments with independent t-tests and 95% confidence intervals.


---

## Summary

Four questions, one thread: **where does this business lose customers, and what is the most precise intervention for each type?**

---

**Case 1 — How many customers ever return?**

64.3% make at least one repeat purchase over the full 12-month window. This looks healthy — but it is a lifetime metric. A customer who returned once in December counts the same as one who returned every month. The number tells us retention *exists*; it says nothing about *when* or *whether the business can influence it*. That is the job of Case 2.

---

**Case 2 — When is the right moment to intervene?**

The 31–60 day window captures the largest incremental return rate: +14.9pp, nearly double the 61–90 day window (+8.6pp). The overall inter-purchase gap distribution is right-skewed — median = 28 days, mean = 45.7 days — meaning most customers who return do so quickly and without prompting. The addressable segment is the slow-return tail: customers who *might* come back if nudged, but won't do so on their own within 30 days.

- A **30-day coupon** expires before the peak return window opens → wasted on customers who haven't yet decided
- A **90-day coupon** removes urgency → customers procrastinate and the lift disappears
- A **45-day window** lands inside the 31–60 day peak while creating a deadline that forces a decision

This is the population-level rationale. Whether the same window applies to all segments is tested in Case 4.

---

**Case 3 — Where in the funnel does the business lose the most customers?**

| Step | Customers remaining | Drop-off |
|------|-------------------|---------|
| 1st purchase | 4,338 | — |
| 1st → 2nd | 2,845 | **−34.4%** ← steepest by ~10pp |
| 2nd → 3rd | 2,010 | −29.3% |
| 3rd → 4th | 1,502 | −25.3% |
| 4th → 5th | 1,114 | −25.8% |

The 1→2 transition loses 34.4% of customers — nearly 10 percentage points worse than any subsequent step. From the 3rd purchase onward, retention stabilises at 70–75% and stays there.

**The problem is activation, not long-term loyalty.** Once a customer has made three purchases, the business is good at keeping them. The failure point is converting a first purchase into a habit. Every intervention should focus here — not on customers who are already returning regularly.

---

**Case 4 — Does the same 45-day coupon work for every segment?**

No — and the reason is more precise than "different customers are different." The critical variable is the **ratio of current recency to historical median gap**. A customer whose recency matches their gap is on schedule. A customer whose recency is 4–5× their gap has structurally disengaged and needs a fundamentally different approach.

The LAG()-based gap analysis reveals this clearly:

| Segment | Median gap | Mean gap | P25–P75 | Current recency | Recency / gap ratio |
|---------|-----------|---------|---------|-----------------|-------------------|
| C2: Whale | 3d | 6.2d | 2–7d | 7d | ~2× (still active) |
| C3: High-value loyal | 14d | 18.5d | 7–25d | 15.5d | ~1× (on schedule) |
| C0: Regular | 43d | 61.8d | 21–81d | 43.7d | ~1× (mid-cycle) |
| C1: At-risk / lapsed | 48d | 59.1d | 21–85d | 248d | **~5× (structurally lapsed)** |

The most important observation in this table: **C0 and C1 have nearly identical gap distributions** (median 43d vs 48d, mean 62d vs 59d). They did not have fundamentally different purchase rhythms when active. What separates them is entirely in the recency column — C0 is mid-cycle, C1 has been gone for 5× their normal interval. Same historical behaviour, completely different current status, completely different intervention needed.

**Recommended tactics by segment:**

| Segment | Tactic | Reasoning |
|---------|--------|-----------|
| C2: Whale (n=13) | Dedicated account management | Recency (7d) < median gap (3d) — still actively purchasing. This is a relationship risk, not an activation problem. One lost Whale is immediately visible in revenue. |
| C3: High-value loyal (n=204) | VIP retention programme | On schedule (recency ≈ gap). High-AOV, established habit. Focus on preventing churn before it happens — early access, exclusive offers, recognition. Not a coupon target. |
| C0: Regular (n=3,054) | **45–60 day post-first-purchase coupon** | Recency (43.7d) sits at the median of their gap distribution (43d) — they are mid-cycle, not yet thinking about returning. A 30-day coupon expires too early. A 45–60 day window arrives when the purchase decision becomes active, and the wide P75 (81d) means a longer window captures the slow-return tail within this segment too. |
| C1: At-risk / lapsed (n=1,067) | **45-day win-back coupon** | Gap (48d median) and recency (248d) ratio = 5×. Structurally lapsed. The 45-day window aligns with their historical purchase rhythm and creates a deadline. Caveat: avg lifespan is only 28 days and avg orders only 1.6, so most C1 customers have a single purchase — the gap estimate comes from the minority who bought more than once and may not represent the full group. Prioritise the 60–90 day lapsed sub-segment first: lowest reactivation cost, most likely to still remember the brand. |

---

### Limitations

1. **No campaign cost data.** Revenue lift estimates assume zero coupon cost. A 10–15% discount reduces margin; actual ROI depends on discount depth and redemption rate — neither available in this dataset.

2. **Single observation window.** The dataset covers exactly one year (Dec 2010 – Dec 2011). Customers labelled "at-risk" may be seasonal buyers whose next purchase falls outside the observation window. Longer historical data would separate structural churn from seasonal absence.

3. **C1 gap estimate is directional, not precise.** Most C1 customers (avg 1.6 orders, avg lifespan 28 days) have only one purchase on record. The 48-day median gap is calculated from the minority with multiple purchases and should be treated as indicative rather than definitive.

4. **Simulation, not experiment.** Coupon lift figures in `03_ab_test_simulation.ipynb` are derived from historical gap distributions, not a real campaign. Statistical significance holds given the assumptions, but the assumptions — control rate and lift magnitude — are estimates.

5. **All purchases treated equally.** High-AOV and low-AOV purchases may have different retention dynamics. Category-level variation is not captured here.

---

### What comes next

`03_ab_test_simulation.ipynb` takes the segment-specific coupon windows from Case 4 and runs simulated A/B tests — C0 and C1 separately — using independent t-tests and 95% confidence intervals. Both results are statistically significant, with an estimated combined revenue lift of **£286,929**.

Note on C0 coupon window: the simulation in `03_ab_test_simulation.ipynb` uses a 30-day window for C0, based on the population-level median gap of 28 days (Case 2). The 45–60 day recommendation above is derived from the segment-level LAG() analysis in Case 4, which shows C0's median gap is 43 days — longer than the population average. The simulation results remain valid as a lower-bound estimate; a 45–60 day window for C0 would likely produce equal or higher lift.
